In [1]:
!pip install -q --upgrade --extra-index-url https://download.pytorch.org/whl/cu121 torch==2.5.1 transformers==4.48.3 accelerate==1.3.0 bitsandbytes==0.45.1 evaluate==0.4.3 rouge-score==0.1.2 pandas==2.2.3 numpy==1.26.4 huggingface-hub==0.27.1 sentencepiece==0.2.0 protobuf==5.29.3

# UdaciHeadline: LLM Inference Optimization Project

## Project Introduction
Large Language Models (LLMs) are transforming content creation, but deploying them efficiently remains a major hurdle. Imagine you're an ML Engineer at a bustling online news portal. Your key task? Automatically generating catchy headlines from article summaries using an LLM. The problem? The current inference process is sluggish, causing publication delays and driving up operational costs. In this project, UdaciHeadline, you'll step into this role and tackle this critical challenge head-on. Your mission is to accelerate the headline generation pipeline significantly by applying state-of-the-art LLM inference optimization techniques. Get ready to dive deep into practical optimization and deployment!

## Project Summary
This project provides hands-on experience in optimizing the inference performance of a pre-trained Large Language Model (like Llama-3.2-1B) for news headline generation. You will bring together concepts of LLM architecture, optimization techniques, and deployment frameworks. Specifically, you will:

1.  **Establish a baseline** inference pipeline and profile its performance.
2.  Implement and evaluate architectural optimizations like **KV-caching**.
3.  Apply model compression techniques like **quantization** and **pruning**.
4.  Configure and benchmark **distributed inference** using Tensor and Pipeline Parallelism.
5.  Apply advanced decoding mechanisms like **speculative decoding**.
6.  Perform comprehensive **benchmarking and analysis** across all stages.
7.  Produce a **final report** summarizing findings and trade-offs.

## Imports and Global Configuration

Let's import the libraries we'll use throughout the project and define some constants like the model name and the prompt template.

In [2]:
import os
import gc
import random
import time
import warnings

import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from evaluate import load as load_metric
from pprint import pprint
import torch.nn.utils.prune as prune
from IPython.display import display, Markdown

from huggingface_hub import login

# Use an environment variable for Hugging Face access. Do not store tokens in notebooks.
_hf_token = os.environ.get("HF_TOKEN")
if _hf_token:
    login(token=_hf_token)
else:
    login()  # interactive

warnings.filterwarnings("ignore")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

# ---- Constants ----
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
TARGET_MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
DATA_PATH = "News_Category_Dataset_v3.json"  # change if needed

SEED = 42
EVAL_N = 100          # use 5 for a quick smoke test
MIN_DESC_CHARS = 30
MAX_NEW_TOKENS = 24   # fixed across experiments so latency comparisons are fair
N_WARMUP = 3
N_ITERS = 20

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _native_bf16_supported():
    """T4/Turing reports bf16 via emulation, which is slow + unfair for benchmarks.
    Only treat bf16 as available when the hardware supports it natively (Ampere+)."""
    if not torch.cuda.is_available():
        return False
    try:
        return torch.cuda.is_bf16_supported(including_emulation=False)
    except TypeError:
        major, _ = torch.cuda.get_device_capability()
        return major >= 8


BF16_OK = _native_bf16_supported()
DTYPE = torch.bfloat16 if BF16_OK else torch.float16

PROMPT = (
    "You are a professional news editor. Write one concise, catchy headline for the "
    "article. Respond with ONLY the headline text: no quotes, no preamble, no explanation."
)

FEWSHOT_PROMPT = (
    "Article: A local bakery is giving away free bread to families in need this winter.\n"
    "Headline: Local Bakery Gives Free Bread To Families This Winter\n\n"
    "Article: Scientists discovered a new species of frog in the Amazon rainforest.\n"
    "Headline: New Frog Species Discovered In The Amazon\n\n"
)

all_metrics = []
metrics_by_name = {}
rouge_metric = load_metric("rouge")


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


set_seed(SEED)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| count:", torch.cuda.device_count())
    print("CUDA:", torch.version.cuda, "| native bf16:", BF16_OK, "| dtype:", DTYPE)
else:
    print("CUDA not available; CPU runs are valid for code checks but too slow for final numbers.")

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: Tesla T4 | count: 4
CUDA: 12.1 | native bf16: False | dtype: torch.float16


## Data Loading

We will use the "News Category Dataset" from Kaggle. The `kagglehub` library makes it easy to download and access. Your task is to implement the function to load and preprocess the data according to the docstring.

In [3]:
def load_news_dataset(path, min_desc_chars=MIN_DESC_CHARS, n_eval=EVAL_N, seed=SEED):
    """Load the local News Category Dataset and return a list of eval records.

    Returns a *list of dicts* (not a HF Dataset) so that the `dataset[:n]`
    slicing in evaluate_model() yields rows, not column names.
    """
    print(f"Loading local dataset: {path}")
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Dataset not found at {path}. Download the News Category Dataset "
            f"(v3) and set DATA_PATH to its .json file."
        )

    # JSON-lines first; fall back to a normal JSON array, then CSV.
    try:
        raw = pd.read_json(path, lines=True)
    except ValueError:
        if path.lower().endswith(".csv"):
            raw = pd.read_csv(path)
        else:
            raw = pd.read_json(path)

    n_raw = len(raw)

    required = {"short_description", "headline"}
    missing = required - set(raw.columns)
    if missing:
        raise KeyError(f"Dataset is missing required columns: {missing}. Found: {list(raw.columns)}")

    raw = raw.dropna(subset=["short_description", "headline"]).copy()
    raw["short_description"] = raw["short_description"].astype(str).str.strip()
    raw["headline"] = raw["headline"].astype(str).str.strip()

    usable = raw[
        (raw["short_description"].str.len() >= min_desc_chars)
        & (raw["headline"].str.len() > 0)
    ].reset_index(drop=True)
    n_usable = len(usable)

    # Deterministic shuffle, then take the eval subset.
    usable = usable.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    eval_df = usable.head(min(n_eval, n_usable))

    records = eval_df[["short_description", "headline"]].to_dict(orient="records")
    print(f"Loaded {n_raw} raw rows -> {n_usable} usable rows -> {len(records)} eval rows")
    if records:
        print("Example description:", records[0]["short_description"])
        print("Example headline:   ", records[0]["headline"])
    return records

# 2. Baseline Performance

Before we can optimize, we need a starting point. Here, you'll establish the baseline performance of the `Llama-3.2-1B` model without any specific optimizations. We will measure latency, throughput, and the quality of the generated headlines using the ROUGE score.

### Your Task: Implement the Evaluation Pipeline
You need to implement the core functions for loading a model, generating a headline, and evaluating performance. These functions will be reused for every optimization technique.

In [4]:
def _model_device(model):
    try:
        return model.device
    except Exception:
        return next(model.parameters()).device


def _move_batch_to_model(batch, model):
    device = _model_device(model)
    return {k: v.to(device) for k, v in batch.items()}


def load_model(model_name, quantization_config=None, device_map=None):
    """Load tokenizer + causal LM. Quantized/distributed loads use device_map;
    plain fp16/bf16 loads are moved to a single GPU when available."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"  # critical for batched decoder-only generation

    kwargs = {"torch_dtype": DTYPE, "low_cpu_mem_usage": True}
    if quantization_config is not None:
        kwargs["quantization_config"] = quantization_config
        kwargs["device_map"] = device_map or "auto"
    elif device_map is not None:
        kwargs["device_map"] = device_map
    else:
        kwargs["attn_implementation"] = "sdpa"

    try:
        model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    except TypeError:
        kwargs.pop("attn_implementation", None)
        model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)

    if quantization_config is None and device_map is None:
        model = model.to(DEVICE)
    model.eval()
    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tokenizer.pad_token_id
    return tokenizer, model


def make_prompt(summary, tokenizer):
    if isinstance(summary, dict):
        summary = summary["short_description"]
    user_text = f"Article: {str(summary).strip()}\n\nHeadline:"
    if getattr(tokenizer, "chat_template", None):
        messages = [
            {"role": "system", "content": PROMPT},
            {"role": "user", "content": user_text},
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return FEWSHOT_PROMPT + user_text


def postprocess_headline(text):
    text = str(text).strip()
    for line in text.splitlines():
        line = line.strip().strip('"').strip("'").strip()
        if not line:
            continue
        for prefix in ("Headline:", "Generated headline:", "Title:"):
            if line.lower().startswith(prefix.lower()):
                line = line[len(prefix):].strip().strip('"').strip("'").strip()
        if line:
            return line
    return text


def build_generation_args(tokenizer, use_cache=True, **extra):
    args = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": False,
        "use_cache": use_cache,
        "pad_token_id": tokenizer.pad_token_id,
    }
    args.update(extra)
    return args


def generate_headline(model, tokenizer, summary, generation_args):
    prompt = make_prompt(summary, tokenizer)
    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        add_special_tokens=(getattr(tokenizer, "chat_template", None) is None),
    )
    enc = _move_batch_to_model(enc, model)
    input_len = enc["input_ids"].shape[1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        with torch.inference_mode():
            output = model.generate(**enc, **generation_args)
        end.record()
        torch.cuda.synchronize()
        latency_s = start.elapsed_time(end) / 1000.0
        peak_mem_mb = torch.cuda.max_memory_allocated() / 1e6
    else:
        start = time.perf_counter()
        with torch.inference_mode():
            output = model.generate(**enc, **generation_args)
        latency_s = time.perf_counter() - start
        peak_mem_mb = np.nan

    new_tokens = output[:, input_len:]
    decoded = tokenizer.decode(new_tokens[0], skip_special_tokens=True)
    headline = postprocess_headline(decoded)
    return headline, {
        "latency_s": latency_s,
        "new_tokens": int(new_tokens.shape[1]),
        "peak_gpu_mem_mb": peak_mem_mb,
    }


def report_metrics(results, latencies, max_new_tokens):
    predictions = [r["prediction"] for r in results]
    references = [r["reference"] for r in results]
    rouge = rouge_metric.compute(predictions=predictions, references=references, use_stemmer=True)

    total_time = max(float(np.sum(latencies)), 1e-9)
    total_tokens = sum(r["new_tokens"] for r in results)
    peak_values = [r["peak_gpu_mem_mb"] for r in results if not pd.isna(r["peak_gpu_mem_mb"])]

    return {
        "latency_mean_s": float(np.mean(latencies)),
        "latency_p50_s": float(np.percentile(latencies, 50)),
        "latency_p99_s": float(np.percentile(latencies, 99)),
        "latency_std_s": float(np.std(latencies)),
        "throughput_samples_s": len(results) / total_time,
        "throughput_tokens_s": total_tokens / total_time,
        "peak_gpu_mem_mb": max(peak_values) if peak_values else np.nan,
        "rouge1": float(rouge["rouge1"]),
        "rouge2": float(rouge["rouge2"]),
        "rougeL": float(rouge["rougeL"]),
        "avg_new_tokens": total_tokens / max(len(results), 1),
        "n_examples": len(results),
    }


def evaluate_model(dataset, model, tokenizer, generation_args, n=20):
    examples = list(dataset[: min(n, len(dataset))])
    if not examples:
        raise ValueError("Dataset is empty after preprocessing; check the dataset path/columns.")

    warmup_example = examples[0]["short_description"]
    for _ in range(N_WARMUP):
        _ = generate_headline(model, tokenizer, warmup_example, generation_args)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    rows, latencies = [], []
    for i, ex in enumerate(examples, start=1):
        pred, info = generate_headline(model, tokenizer, ex["short_description"], generation_args)
        rows.append({
            "idx": i,
            "summary": ex["short_description"],
            "reference": ex["headline"],
            "prediction": pred,
            **info,
        })
        latencies.append(info["latency_s"])
        if i <= 3:
            print("PRED:", pred)
            print(" REF:", ex["headline"])
            print()

    metrics = report_metrics(rows, latencies, MAX_NEW_TOKENS)
    pprint(metrics)
    return metrics, rows


def record_experiment(name, metrics, notes=""):
    row = {
        "Optimization Technique": name,
        "Mean Latency (s)": metrics.get("latency_mean_s", np.nan),
        "P99 Latency (s)": metrics.get("latency_p99_s", np.nan),
        "Throughput (tokens/s)": metrics.get("throughput_tokens_s", np.nan),
        "Throughput (samples/s)": metrics.get("throughput_samples_s", np.nan),
        "Peak GPU Memory (MB)": metrics.get("peak_gpu_mem_mb", np.nan),
        "ROUGE-1 Score": metrics.get("rouge1", np.nan),
        "ROUGE-2 Score": metrics.get("rouge2", np.nan),
        "ROUGE-L Score": metrics.get("rougeL", np.nan),
        "Examples": metrics.get("n_examples", np.nan),
        "Notes": notes,
    }
    all_metrics[:] = [r for r in all_metrics if r["Optimization Technique"] != name]
    all_metrics.append(row)
    metrics_by_name[name] = row
    return row


def record_skip(name, notes):
    return record_experiment(name, {}, notes=notes)

In [5]:
# Establish baseline performance: no KV cache, greedy decoding, fixed output budget.
set_seed(SEED)
free_memory()

news_eval = load_news_dataset(DATA_PATH)
tokenizer, model = load_model(MODEL_NAME)

baseline_args = build_generation_args(tokenizer, use_cache=False)
baseline_metrics, baseline_rows = evaluate_model(
    news_eval, model, tokenizer, baseline_args, n=EVAL_N
)
record_experiment("Baseline (No Cache)", baseline_metrics, notes="use_cache=False")
pd.DataFrame(all_metrics)

Loading local dataset: News_Category_Dataset_v3.json
Loaded 209527 raw rows -> 179533 usable rows -> 100 eval rows
Example description: Some of the most popular animal stories from the last week include: Two good samaritans saved a shark that was choking to
Example headline:    Animal Photos Of The Week: Baby Gibbon Hangs Out With His Mom
PRED: Rescue of the Shark
 REF: Animal Photos Of The Week: Baby Gibbon Hangs Out With His Mom

PRED: Parenting 2.0: Rethinking Our Approach
 REF: 4 Powerful and Effective Alternatives to Punishment

PRED: Loneliness Epidemic
 REF: Bridging the Loneliness Gap: Making a Happier and Healthier Future for the Elderly

{'avg_new_tokens': 10.33,
 'latency_mean_s': 0.20785419464111324,
 'latency_p50_s': 0.1943456497192383,
 'latency_p99_s': 0.4093892617797852,
 'latency_std_s': 0.07923604023759882,
 'n_examples': 100,
 'peak_gpu_mem_mb': 2491.659264,
 'rouge1': 0.12039118944463198,
 'rouge2': 0.02047358772230895,
 'rougeL': 0.11071602072752629,
 'throughput_s

,Optimization Technique,Mean Latency (s),P99 Latency (s),Throughput (tokens/s),Throughput (samples/s),Peak GPU Memory (MB),ROUGE-1 Score,ROUGE-2 Score,ROUGE-L Score,Examples,Notes
0,Baseline (No Cache),0.207854,0.409389,49.698299,4.811065,2491.659264,0.120391,0.020474,0.110716,100,use_cache=False


# 3. Architectural Optimization: KV Caching

**Your Task:** One of the most effective ways to speed up token generation is using a Key-Value (KV) cache. This avoids re-computing attention scores for tokens that are already part of the sequence. Enable the `use_cache` flag in the generation arguments and re-run the evaluation. Observe the impact on latency and throughput.

In [6]:
# Evaluate the identical model and dataset with KV caching enabled.
kv_args = build_generation_args(tokenizer, use_cache=True)
kv_metrics, kv_rows = evaluate_model(
    news_eval, model, tokenizer, kv_args, n=EVAL_N
)
record_experiment("KV Caching", kv_metrics, notes="use_cache=True")

base = metrics_by_name["Baseline (No Cache)"]
kv = metrics_by_name["KV Caching"]
print("Latency speedup vs baseline: {:.2f}x".format(base["Mean Latency (s)"] / kv["Mean Latency (s)"]))
print("ROUGE-L delta vs baseline: {:+.4f}".format(kv["ROUGE-L Score"] - base["ROUGE-L Score"]))
pd.DataFrame(all_metrics)

PRED: Rescue of the Shark
 REF: Animal Photos Of The Week: Baby Gibbon Hangs Out With His Mom

PRED: Parenting 2.0: Rethinking Our Approach
 REF: 4 Powerful and Effective Alternatives to Punishment

PRED: Loneliness Epidemic
 REF: Bridging the Loneliness Gap: Making a Happier and Healthier Future for the Elderly

{'avg_new_tokens': 10.43,
 'latency_mean_s': 0.1764897863006592,
 'latency_p50_s': 0.16921942138671875,
 'latency_p99_s': 0.35235240142822266,
 'latency_std_s': 0.06460078491190711,
 'n_examples': 100,
 'peak_gpu_mem_mb': 2495.105024,
 'rouge1': 0.11993763540119708,
 'rouge2': 0.02047358772230895,
 'rougeL': 0.11029671595660284,
 'throughput_samples_s': 5.6660502625146245,
 'throughput_tokens_s': 59.096904238027534}
Latency speedup vs baseline: 1.18x
ROUGE-L delta vs baseline: -0.0004


,Optimization Technique,Mean Latency (s),P99 Latency (s),Throughput (tokens/s),Throughput (samples/s),Peak GPU Memory (MB),ROUGE-1 Score,ROUGE-2 Score,ROUGE-L Score,Examples,Notes
0,Baseline (No Cache),0.207854,0.409389,49.698299,4.811065,2491.659264,0.120391,0.020474,0.110716,100,use_cache=False
1,KV Caching,0.176490,0.352352,59.096904,5.666050,2495.105024,0.119938,0.020474,0.110297,100,use_cache=True


# 4. Model Compression: Pruning

**Your Task:** Pruning removes redundant model weights, which can reduce model size and potentially speed up inference. Here, you will implement unstructured, magnitude-based pruning by creating a function that applies it to the model's linear layers and then evaluating the result.

In [7]:
def prune_model_weights(model, amount=0.3):
    """Apply L1 unstructured pruning to linear-layer weights except lm_head."""
    pruned_layers = 0
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear) and "lm_head" not in name:
            prune.l1_unstructured(module, name="weight", amount=amount)
            prune.remove(module, "weight")  # make pruning permanent
            pruned_layers += 1
    return pruned_layers


def global_linear_sparsity(model):
    zeros, total = 0, 0
    for module in model.modules():
        if isinstance(module, torch.nn.Linear):
            weight = module.weight.detach()
            zeros += int((weight == 0).sum().item())
            total += weight.numel()
    return zeros / max(total, 1)


# Evaluate a freshly loaded 30% pruned model
try:
    del model
except NameError:
    pass
free_memory()

pruned_tokenizer, pruned_model = load_model(MODEL_NAME)
pruned_count = prune_model_weights(pruned_model, amount=0.30)
print(f"Pruned {pruned_count} linear layers; global linear sparsity = {global_linear_sparsity(pruned_model):.1%}")

pruned_args = build_generation_args(pruned_tokenizer, use_cache=True)
pruned_metrics, pruned_rows = evaluate_model(
    news_eval, pruned_model, pruned_tokenizer, pruned_args, n=EVAL_N
)
record_experiment(
    "Pruning (30%)",
    pruned_metrics,
    notes="30% unstructured L1 pruning; dense GPU kernels usually do not exploit the zeros.",
)

del pruned_model
free_memory()
pd.DataFrame(all_metrics)


Pruned 112 linear layers; global linear sparsity = 23.6%
PRED: Shark Rescue: Two Good Samaritans Save the Day
 REF: Animal Photos Of The Week: Baby Gibbon Hangs Out With His Mom

PRED: Parenting 2.0: Revisiting Our Actions for a Healthier, Happier Child
 REF: 4 Powerful and Effective Alternatives to Punishment

PRED: Loneliness Epidemic
 REF: Bridging the Loneliness Gap: Making a Happier and Healthier Future for the Elderly

{'avg_new_tokens': 9.77,
 'latency_mean_s': 0.16664697151184082,
 'latency_p50_s': 0.1367695007324219,
 'latency_p99_s': 0.38839541168212893,
 'latency_std_s': 0.0764590694896535,
 'n_examples': 100,
 'peak_gpu_mem_mb': 2495.105024,
 'rouge1': 0.09719595463875644,
 'rouge2': 0.017331202046035804,
 'rougeL': 0.08804170388435092,
 'throughput_samples_s': 6.0007091093698435,
 'throughput_tokens_s': 58.626927998543366}


,Optimization Technique,Mean Latency (s),P99 Latency (s),Throughput (tokens/s),Throughput (samples/s),Peak GPU Memory (MB),ROUGE-1 Score,ROUGE-2 Score,ROUGE-L Score,Examples,Notes
0,Baseline (No Cache),0.207854,0.409389,49.698299,4.811065,2491.659264,0.120391,0.020474,0.110716,100,use_cache=False
1,KV Caching,0.176490,0.352352,59.096904,5.666050,2495.105024,0.119938,0.020474,0.110297,100,use_cache=True
2,Pruning (30%),0.166647,0.388395,58.626928,6.000709,2495.105024,0.097196,0.017331,0.088042,100,30% unstructured L1 pruning; dense GPU kernels...


# 5. Model Compression: Quantization

**Your Task:** Quantization reduces the precision of model weights (e.g., from 16-bit to 4-bit), significantly cutting down memory usage and often speeding up inference. You will define a 4-bit quantization configuration and use it to load and evaluate a new model.

In [8]:
def _peak_mem_mb_all_devices():
    if not torch.cuda.is_available():
        return float("nan")

    return sum(
        torch.cuda.max_memory_allocated(d)
        for d in range(torch.cuda.device_count())
    ) / 1e6


def _reset_peak_all_devices():
    if torch.cuda.is_available():
        for d in range(torch.cuda.device_count()):
            torch.cuda.reset_peak_memory_stats(d)


if not torch.cuda.is_available():
    record_skip(
        "Quantization (4-bit)",
        "Skipped: bitsandbytes 4-bit requires CUDA."
    )

else:
    try:
        free_memory()

        quant_config_4bit = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=DTYPE,
            bnb_4bit_use_double_quant=True,
        )

        quant_tokenizer, quant_model = load_model(
            MODEL_NAME,
            quantization_config=quant_config_4bit,
            device_map="auto",
        )

        footprint_mb = quant_model.get_memory_footprint() / 1e6
        print(
            f"4-bit static footprint: {footprint_mb:.1f} MB "
            f"({footprint_mb / 1024:.3f} GB)"
        )

        quant_args = build_generation_args(
            quant_tokenizer,
            use_cache=True,
        )

        sanity_pred, _ = generate_headline(
            quant_model,
            quant_tokenizer,
            news_eval[0]["short_description"],
            quant_args,
        )

        print("Sanity-check headline:", sanity_pred)

        assert sanity_pred and len(sanity_pred) > 3, (
            "Quantized model generated an empty headline."
        )

        # Reset peak counters on all devices, then evaluate.
        _reset_peak_all_devices()

        quant_metrics, quant_rows = evaluate_model(
            news_eval,
            quant_model,
            quant_tokenizer,
            quant_args,
            n=EVAL_N,
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        peak_all_mb = _peak_mem_mb_all_devices()

        print(
            f"Peak GPU memory across all "
            f"{torch.cuda.device_count()} device(s): {peak_all_mb:.1f} MB"
        )

        # Overwrite the misleading single-device peak with the all-device figure.
        quant_metrics["peak_gpu_mem_mb"] = peak_all_mb

        record_experiment(
            "Quantization (4-bit)",
            quant_metrics,
            notes=(
                f"NF4 4-bit + double quant; "
                f"static footprint {footprint_mb:.0f} MB; "
                f"peak summed across all GPUs (device_map='auto')."
            ),
        )

    except Exception as exc:
        record_skip(
            "Quantization (4-bit)",
            f"Skipped/failed: {type(exc).__name__}: {exc}",
        )
        print("4-bit quantization failed:", repr(exc))

    finally:
        try:
            del quant_model
        except NameError:
            pass

        free_memory()


pd.DataFrame(all_metrics)

4-bit static footprint: 1012.0 MB (0.988 GB)
Sanity-check headline: Rescue Efforts Continue
PRED: Rescue Efforts Continue
 REF: Animal Photos Of The Week: Baby Gibbon Hangs Out With His Mom

PRED: Parenting in the Mirror
 REF: 4 Powerful and Effective Alternatives to Punishment

PRED: Loneliness Epidemic
 REF: Bridging the Loneliness Gap: Making a Happier and Healthier Future for the Elderly

{'avg_new_tokens': 12.26,
 'latency_mean_s': 0.3231252072906495,
 'latency_p50_s': 0.31653610229492185,
 'latency_p99_s': 0.571697579956055,
 'latency_std_s': 0.11046631328792204,
 'n_examples': 100,
 'peak_gpu_mem_mb': 8.51968,
 'rouge1': 0.13056168374644955,
 'rouge2': 0.02201245459906451,
 'rougeL': 0.11510169054334485,
 'throughput_samples_s': 3.0947755775070345,
 'throughput_tokens_s': 37.94194858023624}
Peak GPU memory across all 4 device(s): 1090.8 MB


,Optimization Technique,Mean Latency (s),P99 Latency (s),Throughput (tokens/s),Throughput (samples/s),Peak GPU Memory (MB),ROUGE-1 Score,ROUGE-2 Score,ROUGE-L Score,Examples,Notes
0,Baseline (No Cache),0.207854,0.409389,49.698299,4.811065,2491.659264,0.120391,0.020474,0.110716,100,use_cache=False
1,KV Caching,0.176490,0.352352,59.096904,5.666050,2495.105024,0.119938,0.020474,0.110297,100,use_cache=True
2,Pruning (30%),0.166647,0.388395,58.626928,6.000709,2495.105024,0.097196,0.017331,0.088042,100,30% unstructured L1 pruning; dense GPU kernels...
3,Quantization (4-bit),0.323125,0.571698,37.941949,3.094776,1090.817024,0.130562,0.022012,0.115102,100,NF4 4-bit + double quant; static footprint 101...


# 6. Distributed Inference (Multi-GPU)

**Your Task:** If you have multiple GPUs, you can split the model across them to reduce the memory burden on a single GPU and potentially improve latency. We will explore two common techniques: Tensor Parallelism and Pipeline Parallelism.

*Note: This section requires a multi-GPU environment.*

### Tensor Parallelism
In this notebook, `device_map="auto"` is used as an Accelerate multi GPU layer sharding comparison. This lowers the memory held by each GPU, but it should not be interpreted as full tensor parallel execution of every matrix multiplication.

### Pipeline Parallelism
In this notebook, `device_map="balanced"` is used as a practical layer sharding comparison. It resembles a pipeline placement in that different blocks live on different GPUs, but it is not a full DeepSpeed style pipeline parallel runtime.

In [9]:
# Tensor-parallel-style distributed inference via Accelerate device_map="auto".
# Run this on the SageMaker multi-GPU workspace. On a single GPU, we record a skip row.
if torch.cuda.device_count() < 2:
    record_skip(
        "Tensor Parallelism",
        "Skipped on this machine: requires >=2 visible GPUs; run on SageMaker multi-GPU.",
    )
    print("Single-GPU environment detected; tensor parallelism cell skipped.")
else:
    try:
        free_memory()
        tp_tokenizer, tp_model = load_model(MODEL_NAME, device_map="auto")
        print("device_map='auto':")
        pprint(getattr(tp_model, "hf_device_map", "No hf_device_map available"))
        tp_args = build_generation_args(tp_tokenizer, use_cache=True)
        tp_metrics, tp_rows = evaluate_model(news_eval, tp_model, tp_tokenizer, tp_args, n=EVAL_N)
        record_experiment(
            "Tensor Parallelism",
            tp_metrics,
            notes="Accelerate device_map='auto' on multi-GPU; run on SageMaker.",
        )
    finally:
        try:
            del tp_model
        except NameError:
            pass
        free_memory()

pd.DataFrame(all_metrics)


device_map='auto':
{'lm_head': 0,
 'model.embed_tokens': 0,
 'model.layers.0': 0,
 'model.layers.1': 0,
 'model.layers.10': 2,
 'model.layers.11': 2,
 'model.layers.12': 2,
 'model.layers.13': 2,
 'model.layers.14': 2,
 'model.layers.15': 2,
 'model.layers.2': 1,
 'model.layers.3': 1,
 'model.layers.4': 1,
 'model.layers.5': 1,
 'model.layers.6': 1,
 'model.layers.7': 1,
 'model.layers.8': 1,
 'model.layers.9': 2,
 'model.norm': 2,
 'model.rotary_emb': 2}
PRED: Rescue of the Shark
 REF: Animal Photos Of The Week: Baby Gibbon Hangs Out With His Mom

PRED: Parenting 2.0: Rethinking Our Approach
 REF: 4 Powerful and Effective Alternatives to Punishment

PRED: Loneliness Epidemic
 REF: Bridging the Loneliness Gap: Making a Happier and Healthier Future for the Elderly

{'avg_new_tokens': 10.43,
 'latency_mean_s': 0.2510327775573731,
 'latency_p50_s': 0.240909294128418,
 'latency_p99_s': 0.5044572552490234,
 'latency_std_s': 0.092860162706455,
 'n_examples': 100,
 'peak_gpu_mem_mb': 788.0555

,Optimization Technique,Mean Latency (s),P99 Latency (s),Throughput (tokens/s),Throughput (samples/s),Peak GPU Memory (MB),ROUGE-1 Score,ROUGE-2 Score,ROUGE-L Score,Examples,Notes
0,Baseline (No Cache),0.207854,0.409389,49.698299,4.811065,2491.659264,0.120391,0.020474,0.110716,100,use_cache=False
1,KV Caching,0.176490,0.352352,59.096904,5.666050,2495.105024,0.119938,0.020474,0.110297,100,use_cache=True
2,Pruning (30%),0.166647,0.388395,58.626928,6.000709,2495.105024,0.097196,0.017331,0.088042,100,30% unstructured L1 pruning; dense GPU kernels...
3,Quantization (4-bit),0.323125,0.571698,37.941949,3.094776,1090.817024,0.130562,0.022012,0.115102,100,NF4 4-bit + double quant; static footprint 101...
4,Tensor Parallelism,0.251033,0.504457,41.548359,3.983544,788.055552,0.119938,0.020474,0.110297,100,Accelerate device_map='auto' on multi-GPU; run...


In [10]:
# Pipeline-parallel-style distributed inference via Accelerate device_map="balanced".
# This is layer sharding, not full DeepSpeed pipeline parallelism, but it satisfies the
# starter's requested comparison and documents the trade-off honestly.
if torch.cuda.device_count() < 2:
    record_skip(
        "Pipeline Parallelism",
        "Skipped on this machine: requires >=2 visible GPUs; run on SageMaker multi-GPU.",
    )
    print("Single-GPU environment detected; pipeline parallelism cell skipped.")
else:
    try:
        free_memory()
        pp_tokenizer, pp_model = load_model(MODEL_NAME, device_map="balanced")
        print("device_map='balanced':")
        pprint(getattr(pp_model, "hf_device_map", "No hf_device_map available"))
        pp_args = build_generation_args(pp_tokenizer, use_cache=True)
        pp_metrics, pp_rows = evaluate_model(news_eval, pp_model, pp_tokenizer, pp_args, n=EVAL_N)
        record_experiment(
            "Pipeline Parallelism",
            pp_metrics,
            notes="Accelerate device_map='balanced' layer sharding; not true DeepSpeed pipeline parallelism.",
        )
    finally:
        try:
            del pp_model
        except NameError:
            pass
        free_memory()

pd.DataFrame(all_metrics)


device_map='balanced':
{'lm_head': 0,
 'model.embed_tokens': 0,
 'model.layers.0': 0,
 'model.layers.1': 0,
 'model.layers.10': 2,
 'model.layers.11': 2,
 'model.layers.12': 2,
 'model.layers.13': 2,
 'model.layers.14': 2,
 'model.layers.15': 2,
 'model.layers.2': 1,
 'model.layers.3': 1,
 'model.layers.4': 1,
 'model.layers.5': 1,
 'model.layers.6': 1,
 'model.layers.7': 1,
 'model.layers.8': 1,
 'model.layers.9': 2,
 'model.norm': 2,
 'model.rotary_emb': 2}
PRED: Rescue of the Shark
 REF: Animal Photos Of The Week: Baby Gibbon Hangs Out With His Mom

PRED: Parenting 2.0: Rethinking Our Approach
 REF: 4 Powerful and Effective Alternatives to Punishment

PRED: Loneliness Epidemic
 REF: Bridging the Loneliness Gap: Making a Happier and Healthier Future for the Elderly

{'avg_new_tokens': 10.43,
 'latency_mean_s': 0.25160511032104493,
 'latency_p50_s': 0.2412593002319336,
 'latency_p99_s': 0.5041340188598632,
 'latency_std_s': 0.09299738633140804,
 'n_examples': 100,
 'peak_gpu_mem_mb': 

,Optimization Technique,Mean Latency (s),P99 Latency (s),Throughput (tokens/s),Throughput (samples/s),Peak GPU Memory (MB),ROUGE-1 Score,ROUGE-2 Score,ROUGE-L Score,Examples,Notes
0,Baseline (No Cache),0.207854,0.409389,49.698299,4.811065,2491.659264,0.120391,0.020474,0.110716,100,use_cache=False
1,KV Caching,0.176490,0.352352,59.096904,5.666050,2495.105024,0.119938,0.020474,0.110297,100,use_cache=True
2,Pruning (30%),0.166647,0.388395,58.626928,6.000709,2495.105024,0.097196,0.017331,0.088042,100,30% unstructured L1 pruning; dense GPU kernels...
3,Quantization (4-bit),0.323125,0.571698,37.941949,3.094776,1090.817024,0.130562,0.022012,0.115102,100,NF4 4-bit + double quant; static footprint 101...
4,Tensor Parallelism,0.251033,0.504457,41.548359,3.983544,788.055552,0.119938,0.020474,0.110297,100,Accelerate device_map='auto' on multi-GPU; run...
5,Pipeline Parallelism,0.251605,0.504134,41.453848,3.974482,788.055552,0.119938,0.020474,0.110297,100,Accelerate device_map='balanced' layer shardin...


# 7. Advanced Decoding: Speculative Decoding

**Your Task:** Speculative decoding uses a smaller, faster "draft" model to generate several candidate tokens. A larger, more accurate "target" model then verifies these tokens in a single forward pass. This can significantly speed up generation if the draft model is a good predictor. You will load a larger target model and a smaller draft model, benchmark the target model alone, and then benchmark it with assistance from the draft model.

In [11]:
# Speculative decoding: compare 3B target alone vs 3B target assisted by the 1B draft.
# This benchmark is separate from the 1B baseline: it asks whether assistance speeds up the
# larger target model while preserving target-model quality.
if not torch.cuda.is_available():
    record_skip("Speculative Decoding", "Skipped: target + draft benchmark is intended for a CUDA GPU.")
else:
    draft_model = None
    target_model = None
    try:
        free_memory()
        draft_tokenizer, draft_model = load_model(MODEL_NAME)
        target_tokenizer, target_model = load_model(TARGET_MODEL_NAME)
        print("Draft model:", MODEL_NAME)
        print("Target model:", TARGET_MODEL_NAME)

        target_args = build_generation_args(target_tokenizer, use_cache=True)
        target_metrics, target_rows = evaluate_model(
            news_eval, target_model, target_tokenizer, target_args, n=EVAL_N
        )
        record_experiment("Speculative Target Alone", target_metrics, notes="3B target, no assistant")

        assisted_args = build_generation_args(
            target_tokenizer, use_cache=True, assistant_model=draft_model
        )
        assisted_metrics, assisted_rows = evaluate_model(
            news_eval, target_model, target_tokenizer, assisted_args, n=EVAL_N
        )
        record_experiment(
            "Speculative Decoding",
            assisted_metrics,
            notes="3B target assisted by 1B draft via assistant_model.",
        )

        target_row = metrics_by_name["Speculative Target Alone"]
        assisted_row = metrics_by_name["Speculative Decoding"]
        print("Speculative speedup vs target alone: {:.2f}x".format(
            target_row["Mean Latency (s)"] / assisted_row["Mean Latency (s)"]
        ))
        print("ROUGE-L delta vs target alone: {:+.4f}".format(
            assisted_row["ROUGE-L Score"] - target_row["ROUGE-L Score"]
        ))
    except Exception as exc:
        record_skip("Speculative Decoding", f"Skipped/failed: {type(exc).__name__}: {exc}")
        print("Speculative decoding failed:", repr(exc))
    finally:
        try:
            del target_model
        except Exception:
            pass
        try:
            del draft_model
        except Exception:
            pass
        free_memory()

pd.DataFrame(all_metrics)


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.92it/s]


Draft model: meta-llama/Llama-3.2-1B-Instruct
Target model: meta-llama/Llama-3.2-3B-Instruct
PRED: Heroic Duo Saves Choking Shark
 REF: Animal Photos Of The Week: Baby Gibbon Hangs Out With His Mom

PRED: Parenting Reflections: The Key to Raising Happy, Healthy Kids
 REF: 4 Powerful and Effective Alternatives to Punishment

PRED: Loneliness Epidemic Hits Home: One in Five Struggle with Enduring Isolation
 REF: Bridging the Loneliness Gap: Making a Happier and Healthier Future for the Elderly



From v4.47 onwards, when a model cache is to be returned, `generate` will return a `Cache` instance instead by default (as opposed to the legacy tuple of tuples format). If you want to keep returning the legacy format, please set `return_legacy_cache=True`.


{'avg_new_tokens': 12.52,
 'latency_mean_s': 0.4149798533630371,
 'latency_p50_s': 0.39896351623535153,
 'latency_p99_s': 0.6188119934082035,
 'latency_std_s': 0.11049015930678699,
 'n_examples': 100,
 'peak_gpu_mem_mb': 8933.871104,
 'rouge1': 0.1341886528452653,
 'rouge2': 0.021106390685338053,
 'rougeL': 0.11702949668554069,
 'throughput_samples_s': 2.409755538481935,
 'throughput_tokens_s': 30.170139341793828}
PRED: Heroic Duo Saves Choking Shark
 REF: Animal Photos Of The Week: Baby Gibbon Hangs Out With His Mom

PRED: Parenting Reflections: The Key to Raising Happy, Healthy Kids
 REF: 4 Powerful and Effective Alternatives to Punishment

PRED: Loneliness Epidemic Hits Home: One in Five Struggle with Enduring Isolation
 REF: Bridging the Loneliness Gap: Making a Happier and Healthier Future for the Elderly

{'avg_new_tokens': 12.59,
 'latency_mean_s': 0.5406718083190918,
 'latency_p50_s': 0.5334083251953126,
 'latency_p99_s': 0.8938279760742189,
 'latency_std_s': 0.1498948998663576

,Optimization Technique,Mean Latency (s),P99 Latency (s),Throughput (tokens/s),Throughput (samples/s),Peak GPU Memory (MB),ROUGE-1 Score,ROUGE-2 Score,ROUGE-L Score,Examples,Notes
0,Baseline (No Cache),0.207854,0.409389,49.698299,4.811065,2491.659264,0.120391,0.020474,0.110716,100,use_cache=False
1,KV Caching,0.176490,0.352352,59.096904,5.666050,2495.105024,0.119938,0.020474,0.110297,100,use_cache=True
2,Pruning (30%),0.166647,0.388395,58.626928,6.000709,2495.105024,0.097196,0.017331,0.088042,100,30% unstructured L1 pruning; dense GPU kernels...
3,Quantization (4-bit),0.323125,0.571698,37.941949,3.094776,1090.817024,0.130562,0.022012,0.115102,100,NF4 4-bit + double quant; static footprint 101...
4,Tensor Parallelism,0.251033,0.504457,41.548359,3.983544,788.055552,0.119938,0.020474,0.110297,100,Accelerate device_map='auto' on multi-GPU; run...
5,Pipeline Parallelism,0.251605,0.504134,41.453848,3.974482,788.055552,0.119938,0.020474,0.110297,100,Accelerate device_map='balanced' layer shardin...
6,Speculative Target Alone,0.414980,0.618812,30.170139,2.409756,8933.871104,0.134189,0.021106,0.117029,100,"3B target, no assistant"
7,Speculative Decoding,0.540672,0.893828,23.285845,1.849551,8963.062272,0.135017,0.021106,0.117909,100,3B target assisted by 1B draft via assistant_m...


# 8. Final Report and Analysis

**Your Task:** Consolidate your findings into a summary report. 

1.  Fill in the Markdown table below with the **Latency**, **Throughput**, and **ROUGE scores** for each optimization technique you implemented.
2. Compile the final Project Report in PDF format:
    *   Document the entire process, detailing the methodology, techniques, and libraries used.
    *   Present the final benchmark results clearly.
    *   Provide a thorough analysis of the trade-offs between performance, resources, and quality for each optimization step.
    *   Conclude with recommendations for the most effective optimization strategy for this specific headline generation task, supported by your data.

Some example questions for discussing the trade-offs:
    *   Which method gave the best performance improvement?
    *   Did any methods significantly hurt the ROUGE score (quality)?
    *   Which optimization would you recommend for deployment in a production environment at the news portal, and why? Consider factors like cost, complexity, and performance.

## Performance Comparison

| Optimization Technique | Mean Latency (s) | P99 Latency (s) | Throughput (tokens/s) | Peak GPU Memory (MB) | ROUGE-1 Score | ROUGE-2 Score | ROUGE-L Score |
|------------------------|------------------:|----------------:|----------------------:|---------------------:|--------------:|--------------:|--------------:|
| Baseline (No Cache) | 0.208 | 0.409 | 49.7 | 2492 | 0.120 | 0.020 | 0.111 |
| KV Caching | 0.176 | 0.352 | 59.1 | 2495 | 0.120 | 0.020 | 0.110 |
| Pruning (30%) | 0.167 | 0.388 | 58.6 | 2495 | 0.097 | 0.017 | 0.088 |
| Quantization (4-bit) | 0.323 | 0.572 | 37.9 | 1091 | 0.131 | 0.022 | 0.115 |
| Tensor Parallelism | 0.251 | 0.504 | 41.5 | 788 | 0.120 | 0.020 | 0.110 |
| Pipeline Parallelism | 0.252 | 0.504 | 41.5 | 788 | 0.120 | 0.020 | 0.110 |
| Speculative Target Alone | 0.415 | 0.619 | 30.2 | 8934 | 0.134 | 0.021 | 0.117 |
| Speculative Decoding | 0.541 | 0.894 | 23.3 | 8963 | 0.135 | 0.021 | 0.118 |

The stored results show that KV caching is the strongest production default for this workload: it improves latency and throughput while keeping ROUGE effectively unchanged. Pruning appears fast in mean latency, but it damages ROUGE quality and does not reduce memory on dense GPU kernels. Quantization gives the best memory reduction, but it is slower for this small model. The multi-GPU sharding runs reduce per-GPU memory, yet communication overhead makes them slower than the single-GPU cached model. Speculative decoding is slower than the larger target model alone because the output length is short and the draft verification overhead dominates.


In [ ]:
# === Section 8: Final results presentation ===
from IPython.display import display, Markdown

final_df = pd.DataFrame(all_metrics)

order = ["Baseline (No Cache)", "KV Caching", "Pruning (30%)", "Quantization (4-bit)",
         "Tensor Parallelism", "Pipeline Parallelism",
         "Speculative Target Alone", "Speculative Decoding"]
final_df["__o"] = final_df["Optimization Technique"].apply(
    lambda x: order.index(x) if x in order else len(order))
final_df = final_df.sort_values("__o").drop(columns="__o").reset_index(drop=True)

summary_cols = ["Optimization Technique", "Mean Latency (s)", "P99 Latency (s)",
                "Throughput (tokens/s)", "Peak GPU Memory (MB)", "ROUGE-1 Score",
                "ROUGE-2 Score", "ROUGE-L Score"]

display(Markdown("## Performance Comparison"))
display(Markdown(final_df[summary_cols].round({
    "Mean Latency (s)": 3,
    "P99 Latency (s)": 3,
    "Throughput (tokens/s)": 1,
    "Peak GPU Memory (MB)": 0,
    "ROUGE-1 Score": 3,
    "ROUGE-2 Score": 3,
    "ROUGE-L Score": 3,
}).to_markdown(index=False)))

base = metrics_by_name["Baseline (No Cache)"]
kv = metrics_by_name["KV Caching"]
kv_speedup = base["Mean Latency (s)"] / kv["Mean Latency (s)"]

display(Markdown(f"""
### Key takeaways

* **KV caching** is the best production default: **{kv_speedup:.2f}x** lower latency than baseline with effectively unchanged quality.
* **Pruning (30%)** reduced ROUGE and did not reduce memory because dense GPU kernels do not exploit the zeros.
* **4-bit quantization** is mainly a memory optimization here; it cut memory sharply but increased latency on this small model.
* **Tensor and pipeline comparisons** used Accelerate layer sharding rather than full tensor or DeepSpeed pipeline parallelism.
* **Speculative decoding** was slower than the 3B target alone because these headline outputs are short.
"""))
